In [1]:
import glob, json, random
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm

chemin = glob.glob("/kaggle/input/**/arxiv-metadata-oai-snapshot.json", recursive=True)[0]
print("Fichier :", chemin)

# Même table que dans la version 1 : archive arXiv -> un des 8 groupes
ARCHIVES_PAR_GROUPE = {
    "cs":      ["cs", "cmp-lg"],
    "math":    ["math", "q-alg", "alg-geom", "dg-ga", "funct-an"],
    "stat":    ["stat"],
    "eess":    ["eess"],
    "q-bio":   ["q-bio"],
    "q-fin":   ["q-fin"],
    "econ":    ["econ"],
    "physics": ["physics", "cond-mat", "astro-ph", "hep-ph", "hep-th", "hep-ex",
                "hep-lat", "quant-ph", "gr-qc", "math-ph", "nucl-th", "nucl-ex",
                "nlin", "chao-dyn", "solv-int", "patt-sol", "adap-org", "comp-gas",
                "mtrl-th", "supr-con", "chem-ph", "atom-ph", "acc-phys",
                "plasm-ph", "ao-sci", "bayes-an"],
}
GROUPE_PAR_ARCHIVE = {a: g for g, archives in ARCHIVES_PAR_GROUPE.items() for a in archives}
GROUPES = ["cs", "math", "physics", "q-bio", "q-fin", "stat", "eess", "econ"]

def categories_vers_groupes(categories):
    """Exemple : 'math.CO cs.CG' -> {'math', 'cs'}"""
    archives = {cat.split(".")[0] for cat in categories.split()}
    return {GROUPE_PAR_ARCHIVE[a] for a in archives}

print(categories_vers_groupes("cs.LG stat.ML"), categories_vers_groupes("hep-ph"))

Fichier : /kaggle/input/datasets/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json
{'cs', 'stat'} {'physics'}


In [4]:
nb_articles = 0
compteur_groupes = Counter()
inconnues = Counter()

with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Comptage"):
        article = json.loads(ligne)
        nb_articles += 1
        archives = {cat.split(".")[0] for cat in article["categories"].split()}
        inconnues.update(a for a in archives if a not in GROUPE_PAR_ARCHIVE)
        compteur_groupes.update({GROUPE_PAR_ARCHIVE[a] for a in archives if a in GROUPE_PAR_ARCHIVE})

print(f"Nombre total d'articles : {nb_articles:,}")
print("Archives inconnues :", dict(inconnues) if inconnues else "aucune")
for g in GROUPES:
    print(f"  {g:8s} {compteur_groupes[g]:>10,}  ({100 * compteur_groupes[g] / nb_articles:.1f} %)")

Comptage: 0it [00:00, ?it/s]

Nombre total d'articles : 3,252,452
Archives inconnues : aucune
  cs        1,054,794  (32.4 %)
  math        804,844  (24.7 %)
  physics   1,630,154  (50.1 %)
  q-bio        57,684  (1.8 %)
  q-fin        26,638  (0.8 %)
  stat        154,695  (4.8 %)
  eess        143,534  (4.4 %)
  econ         18,210  (0.6 %)


In [3]:
random.seed(42)

TOTAL_VISE = 20_000   # taille de l'échantillon
MINIMUM = 300         # au moins ~300 articles pour les petits groupes

# Même taux pour tous, sauf les petits groupes qu'on remonte juste assez pour avoir ~300 articles
taux_base = TOTAL_VISE / nb_articles
taux = {g: max(taux_base, MINIMUM / compteur_groupes[g]) for g in GROUPES}
print("Taux de tirage :")
for g in GROUPES:
    print(f"  {g:8s} {100 * taux[g]:.2f} %")

echantillon = []
with open(chemin, "r") as f:
    for ligne in tqdm(f, desc="Tirage"):
        article = json.loads(ligne)
        groupes = categories_vers_groupes(article["categories"])
        if random.random() >= max(taux[g] for g in groupes):
            continue                      # pas tiré
        if len(article["abstract"].split()) < 30:
            continue                      # résumé trop court
        resultat = {"id": article["id"], "title": article["title"], "abstract": article["abstract"]}
        for g in GROUPES:
            resultat[g] = 1 if g in groupes else 0
        echantillon.append(resultat)

df = pd.DataFrame(echantillon)
print("\nNombre d'articles tirés :", len(df))
print(pd.DataFrame({"nombre": df[GROUPES].sum(),
                    "%": (100 * df[GROUPES].mean()).round(1)}).sort_values("nombre", ascending=False))
print("\nGroupes par article en moyenne :", round(df[GROUPES].sum(axis=1).mean(), 2))

df.to_csv("/kaggle/working/arxiv_echantillon.csv", index=False)
print("\nFichier sauvegardé : /kaggle/working/arxiv_echantillon.csv")

Taux de tirage :
  cs       0.61 %
  math     0.61 %
  physics  0.61 %
  q-bio    0.61 %
  q-fin    1.13 %
  stat     0.61 %
  eess     0.61 %
  econ     1.65 %


Tirage: 0it [00:00, ?it/s]


Nombre d'articles tirés : 19982
         nombre     %
physics    9982  50.0
cs         6488  32.5
math       4780  23.9
stat        977   4.9
eess        835   4.2
q-bio       391   2.0
q-fin       368   1.8
econ        347   1.7

Groupes par article en moyenne : 1.21

Fichier sauvegardé : /kaggle/working/arxiv_echantillon.csv
